In [0]:
# Databricks notebook source
# 02_generate_mock_daily_sources

from pyspark.sql import functions as F
from pyspark.sql.types import *

# =========================================================
# CONFIG
# =========================================================
BASE_VOLUME_PATH = "/Volumes/workspace/00_landing/landing"

CUSTOMERS_PATH = f"{BASE_VOLUME_PATH}/customers"
ORDERS_PATH = f"{BASE_VOLUME_PATH}/orders"
ORDER_ITEMS_PATH = f"{BASE_VOLUME_PATH}/order_items"
PRODUCTS_PATH = f"{BASE_VOLUME_PATH}/products"
PAYMENTS_PATH = f"{BASE_VOLUME_PATH}/payments"
CATEGORY_TRANSLATION_PATH = f"{BASE_VOLUME_PATH}/category_translation"

WEB_BASE_PATH = f"{BASE_VOLUME_PATH}/web_events"
ADS_BASE_PATH = f"{BASE_VOLUME_PATH}/api_ads_raw"

dbutils.fs.mkdirs(WEB_BASE_PATH)
dbutils.fs.mkdirs(ADS_BASE_PATH)

run_id_str = spark.sql(
    "select date_format(current_timestamp(),'yyyyMMdd_HHmmss') as run_id"
).first()["run_id"]

# =========================================================
# 1) Read Olist source CSV files from Volume
# =========================================================
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CUSTOMERS_PATH)
    .select("customer_id", "customer_unique_id", "customer_city", "customer_state")
)

orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_PATH)
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDER_ITEMS_PATH)
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    )
)

products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PRODUCTS_PATH)
    .select(
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
)

payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PAYMENTS_PATH)
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_value"),
        F.max("payment_type").alias("payment_type"),
        F.max("payment_installments").alias("payment_installments")
    )
)

category_translation = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CATEGORY_TRANSLATION_PATH)
    .select("product_category_name", "product_category_name_english")
)

# =========================================================
# 2) Build source base
# =========================================================
base = (
    orders.alias("o")
    .join(customers.alias("c"), "customer_id", "inner")
    .join(order_items.alias("oi"), "order_id", "inner")
    .join(products.alias("p"), "product_id", "left")
    .join(payments.alias("pay"), "order_id", "left")
    .join(category_translation.alias("t"), "product_category_name", "left")
    .filter(F.col("order_purchase_timestamp").isNotNull())
)

SAMPLE_ORDER_ITEMS = 1500

base_sample = (
    base.orderBy(F.rand())
        .limit(SAMPLE_ORDER_ITEMS)
        .withColumn(
            "product_category_name_en",
            F.coalesce(F.col("product_category_name_english"), F.lit("unknown"))
        )
)

# =========================================================
# 3) Campaign mapping
# =========================================================
campaign_dim = spark.createDataFrame(
    [
        (0, "cmp_001", "Search - Electronics", "google_ads", "google", "cpc", "search_electronics", "website"),
        (1, "cmp_002", "Social - Home & Living", "facebook_ads", "facebook", "paid_social", "social_home_living", "website"),
        (2, "cmp_003", "Retargeting - Marketplace", "instagram_ads", "instagram", "display", "retargeting_marketplace", "website"),
        (3, "cmp_004", "CRM - High Intent", "email", "newsletter", "email", "crm_high_intent", "website"),
        (4, "cmp_005", "Search - Fashion", "google_ads", "google", "cpc", "search_fashion", "website"),
        (5, "cmp_006", "Social - Baby & Toys", "tiktok_ads", "tiktok", "paid_social", "social_baby_toys", "mobile_app"),
    ],
    [
        "campaign_bucket",
        "campaign_id",
        "campaign_name",
        "referrer",
        "utm_source",
        "utm_medium",
        "utm_campaign",
        "store_channel"
    ]
)

base_enriched = (
    base_sample
    .withColumn(
        "campaign_bucket",
        F.pmod(F.abs(F.xxhash64("product_category_name_en")), F.lit(6))
    )
    .join(campaign_dim, "campaign_bucket", "left")
)

# =========================================================
# 4) Generate web events
# =========================================================
purchase_events = (
    base_enriched
    .withColumn("event_type", F.lit("purchase"))
    .withColumn("quantity", F.lit(1))
    .withColumn("price_at_event", F.col("price").cast("double"))
    .withColumn("event_time_ts", F.to_timestamp("order_purchase_timestamp"))
    .withColumn(
        "event_id",
        F.concat_ws("_", F.lit("evt"), F.lit("purchase"), F.col("order_id"), F.col("order_item_id").cast("string"))
    )
    .select(
        "event_id",
        "event_time_ts",
        F.col("customer_unique_id").alias("user_id"),
        "order_id",
        "product_id",
        "campaign_id",
        "referrer",
        "utm_source",
        "utm_medium",
        "utm_campaign",
        "store_channel",
        "price_at_event",
        "quantity",
        "event_type"
    )
)

add_to_cart_events = (
    base_enriched
    .withColumn("event_type", F.lit("add_to_cart"))
    .withColumn("quantity", F.lit(1))
    .withColumn("price_at_event", F.col("price").cast("double"))
    .withColumn(
        "event_time_ts",
        F.to_timestamp(
            F.from_unixtime(
                F.unix_timestamp(F.to_timestamp("order_purchase_timestamp")) -
                (F.floor(F.rand(11) * 7200) + 300).cast("int")
            )
        )
    )
    .withColumn(
        "event_id",
        F.concat_ws("_", F.lit("evt"), F.lit("cart"), F.col("order_id"), F.col("order_item_id").cast("string"))
    )
    .select(
        "event_id",
        "event_time_ts",
        F.col("customer_unique_id").alias("user_id"),
        "order_id",
        "product_id",
        "campaign_id",
        "referrer",
        "utm_source",
        "utm_medium",
        "utm_campaign",
        "store_channel",
        "price_at_event",
        "quantity",
        "event_type"
    )
)

view_product_events = (
    base_enriched
    .withColumn("event_type", F.lit("view_product"))
    .withColumn("quantity", F.lit(1))
    .withColumn("price_at_event", F.col("price").cast("double"))
    .withColumn(
        "event_time_ts",
        F.to_timestamp(
            F.from_unixtime(
                F.unix_timestamp(F.to_timestamp("order_purchase_timestamp")) -
                (F.floor(F.rand(21) * 86400) + 1800).cast("int")
            )
        )
    )
    .withColumn(
        "event_id",
        F.concat_ws("_", F.lit("evt"), F.lit("view"), F.col("order_id"), F.col("order_item_id").cast("string"))
    )
    .select(
        "event_id",
        "event_time_ts",
        F.col("customer_unique_id").alias("user_id"),
        "order_id",
        "product_id",
        "campaign_id",
        "referrer",
        "utm_source",
        "utm_medium",
        "utm_campaign",
        "store_channel",
        "price_at_event",
        "quantity",
        "event_type"
    )
)

web_events = (
    view_product_events
    .unionByName(add_to_cart_events)
    .unionByName(purchase_events)
    .withColumn("session_id", F.concat(F.lit("sess_"), F.substring(F.md5(F.concat_ws("||", "user_id", "order_id", "product_id")), 1, 12)))
    .withColumn("device", F.when(F.rand(31) < 0.60, F.lit("mobile"))
                           .when(F.rand(32) < 0.85, F.lit("desktop"))
                           .otherwise(F.lit("tablet")))
    .withColumn("browser", F.when(F.rand(41) < 0.55, F.lit("chrome"))
                            .when(F.rand(42) < 0.70, F.lit("safari"))
                            .when(F.rand(43) < 0.85, F.lit("firefox"))
                            .otherwise(F.lit("edge")))
    .withColumn("page_url", F.concat(F.lit("/product/"), F.col("product_id")))
    .withColumn("event_time", F.date_format("event_time_ts", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .select(
        "event_id",
        "event_time",
        "session_id",
        "user_id",
        "event_type",
        "product_id",
        "order_id",
        "quantity",
        "price_at_event",
        "page_url",
        "referrer",
        "utm_source",
        "utm_medium",
        "utm_campaign",
        "campaign_id",
        "device",
        "browser",
        "store_channel"
    )
)

# =========================================================
# 5) Build ads rows
# =========================================================
web_events_ts = (
    web_events
    .withColumn("event_time_ts", F.to_timestamp("event_time", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("date", F.to_date("event_time_ts"))
)

ads_campaigns = (
    web_events_ts.groupBy("date", "campaign_id", "utm_campaign", "utm_source", "utm_medium")
    .agg(
        F.sum(F.when(F.col("event_type") == "view_product", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event_type") == "add_to_cart", 1).otherwise(0)).alias("adds"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases")
    )
    .withColumn(
        "campaign_name",
        F.when(F.col("campaign_id") == "cmp_001", F.lit("Search - Electronics"))
         .when(F.col("campaign_id") == "cmp_002", F.lit("Social - Home & Living"))
         .when(F.col("campaign_id") == "cmp_003", F.lit("Retargeting - Marketplace"))
         .when(F.col("campaign_id") == "cmp_004", F.lit("CRM - High Intent"))
         .when(F.col("campaign_id") == "cmp_005", F.lit("Search - Fashion"))
         .otherwise(F.lit("Social - Baby & Toys"))
    )
    .withColumn(
        "channel",
        F.when(F.col("utm_source") == "google", F.lit("Google"))
         .when(F.col("utm_source") == "facebook", F.lit("Facebook"))
         .when(F.col("utm_source") == "instagram", F.lit("Instagram"))
         .when(F.col("utm_source") == "tiktok", F.lit("TikTok"))
         .otherwise(F.lit("Email"))
    )
    .withColumn("source", F.col("utm_source"))
    .withColumn("medium", F.col("utm_medium"))
    .withColumn("clicks", F.greatest((F.col("views") * 0.28).cast("int") + F.col("adds"), F.col("purchases") + 1))
    .withColumn("impressions", F.greatest((F.col("views") * (8 + F.floor(F.rand(71) * 10))).cast("int"), F.col("clicks") + 10))
    .withColumn("conversions", F.col("purchases").cast("int"))
    .withColumn("cost", F.round((F.col("clicks") * (0.20 + F.rand(81) * 1.30)) + (F.col("conversions") * (0.50 + F.rand(82) * 3.50)), 2))
    .withColumn("currency", F.lit("USD"))
    .select(
        "date",
        "campaign_id",
        "campaign_name",
        "channel",
        "source",
        "medium",
        "impressions",
        "clicks",
        "conversions",
        "cost",
        "currency"
    )
)

# =========================================================
# 6. Write landing files
# =========================================================
web_output_path = f"{WEB_BASE_PATH}/{run_id_str}"
ads_output_path = f"{ADS_BASE_PATH}/{run_id_str}"

(
    web_events
    .repartition(16, "campaign_id")
    .write
    .mode("overwrite")
    .option("maxRecordsPerFile", 200000)
    .json(web_output_path)
)

(
    ads_campaigns
    .repartition(4)
    .write
    .mode("overwrite")
    .option("maxRecordsPerFile", 50000)
    .json(ads_output_path)
)

print(f"Generated web events path: {web_output_path}")
print(f"Generated ads path       : {ads_output_path}")
print(f"Web event rows           : {web_events.count()}")
print(f"Ads rows                 : {ads_campaigns.count()}")